# ETL Code Examples
### CSCI3020 Summer 2026

> **Today's goal.** Take a large, messy CSV of horror films and turn it into clean, structured data sitting in SQL Server -- ready for the data warehouse we design on Thursday.

> **About the data.** These are real films: the title, release year, director, country, and studio are accurate. What has been scrambled is the *formatting* -- dates in five different layouts, money with and without symbols, studios spelled several ways -- plus some invented figures (budget, box office, votes, scores) that stand in for data a studio would not publish. So you can sanity-check your work against films you know.

> **Why Python instead of SSIS?** SSIS (a Microsoft Tool) hides the logic inside drag-and-drop boxes and only runs on Windows in Visual Studio. Writing the pipeline in Python means you can *see* every transformation, run it on any OS, put it in version control, and reuse the same skills when we do Python database integration. The concepts are identicaL!

## Setup

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("pandas", pd.__version__)

### So whate are these imports?
- `pandas` gives us **`DataFrames`**
    - These are in-memory tables we can filter, clean, and reshape
- `numpy` provides the fast numeric foundation pandas is built on; we mostly use it here for `np.nan`, its marker for a missing value

You will use these a LOT in CSCI-4047 (Data Analytics & Visualization) in the fall.

# Part 1: Extracting

All we want tot do heer is get the data in -- we're not judging or making descisions yet.

In [ ]:
# Read EVERYTHING as text first
# If you let pandas guess your types now, it will silently mangle the messy columns
# this creates a lot of peoblrms later
raw = pd.read_csv("horror_movies_raw.csv", dtype=str)

print(f"Extracted {len(raw):,} rows and {len(raw.columns)} columns")
raw.head(10)

> **Why `dtype=str`?** Because a column like `budget_usd` contains `$1,200,000`, `USD 50000`, and blanks. If pandas tries to infer a type it will either fail or quietly turn the whole column into text anyway -- but *after* possibly mis-parsing some values. Read raw, convert deliberately.

## 1.1 Profile before you clean

Never start transforming until you know what you're dealing with. This step is know na s **data profiling**.

In [ ]:
print("=== Shape ===")
print(raw.shape)

print("\n=== Missing values per column ===")
missing = raw.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

print("\n=== Missing as a percentage ===")
print((missing[missing > 0] / len(raw) * 100).round(1))

In [ ]:
# How many exact duplicate records (ignoring the surrogate record_id)?
dupes = raw.drop(columns=["record_id"]).duplicated().sum()
print(f"Exact duplicate rows: {dupes}")

# What do the messy columns actually look like?
print("\nDate formats present:")
print(raw["release_date"].head(10).tolist())

print("\nRuntime formats present:")
print(raw["runtime"].dropna().head(10).tolist())

print("\nMoney formats present:")
print(raw["box_office_usd"].dropna().head(10).tolist())

In [ ]:
# Categorical columns: how many distinct values, and are they consistent?
print("Distinct studios:", raw["studio"].nunique())
print(sorted(raw["studio"].dropna().unique(), reverse=True)[:20])

print("\nDistinct countries:", raw["country"].nunique())
print(sorted(raw["country"].dropna().unique()))

> **Look at this output carefully.** `Universal`, `universal`, `UNIVERSAL`, `Universal Pictures`, and `Universal  ` are all the *same studio* -- but to a database they are five different values. Every `GROUP BY studio` you ever run would be wrong. This is the single most common real-world data problem, and fixing it is called **standardization**.

---

# Part 2: Trasnforming

Now it is time for some real work. We'll go one problem at a time, and keep the raw frame untouched so we can always start over. If this feels like it could get messy or time-consuming, you're exactly right! It is extremely valuable and time well-spent if you value good data, however.


In [ ]:
df = raw.copy()   # never mutate your extract

## 2.1 Trim whitespace and fix casing


In [ ]:
# Strip leading/trailing spaces from every text column.
# (We read everything as text, so this is every column.)
for c in df.columns:
    df[c] = df[c].str.strip()

# Normalize titles to title case
df["title"] = df["title"].str.title()

df[["title", "studio", "country"]].head(10)

## 2.2 Standardize categorical values

This is the fix for the `Universal / universal / UNIVERSAL` problem. We build an explicit **mapping** from messy values to canonical ones.


In [ ]:
studio_map = {
    "universal": "Universal", "universal pictures": "Universal",
    "hammer": "Hammer Films", "hammer films": "Hammer Films",
    "toho": "Toho", "toho co.": "Toho",
    "a24": "A24", "a-24": "A24",
    "blumhouse": "Blumhouse", "blumhouse productions": "Blumhouse",
    "warner bros": "Warner Bros.", "warner bros.": "Warner Bros.",
    "paramount": "Paramount", "paramount pictures": "Paramount",
    "new line": "New Line Cinema", "new line cinema": "New Line Cinema",
    "neon": "Neon",
}

def clean_studio(s):
    if pd.isna(s):
        return None
    key = s.strip().lower()
    return studio_map.get(key, s.strip())   # fall back to the original if unmapped

df["studio"] = df["studio"].apply(clean_studio)

print("Distinct studios after standardizing:", df["studio"].nunique())
print(sorted(df["studio"].dropna().unique(), reverse=True))

In [ ]:
country_map = {
    "usa": "USA", "u.s.a.": "USA", "united states": "USA", "us": "USA",
    "uk": "UK", "u.k.": "UK", "united kingdom": "UK", "england": "UK",
    "italy": "Italy", "japan": "Japan", "nippon": "Japan",
    "south korea": "South Korea", "korea": "South Korea", "s. korea": "South Korea",
    "france": "France", "canada": "Canada",
}

df["country"] = df["country"].apply(
    lambda s: country_map.get(s.strip().lower(), s.strip()) if pd.notna(s) else None
)

print("Distinct countries after standardizing:", df["country"].nunique())
print(sorted(df["country"].dropna().unique()))

> Compare the before/after counts. We just collapsed several spurious "categories" into the real ones. **Every aggregate query downstream is now correct.**
>
> **Note what we did *not* do.** Our map only covers the studios we knew were messy. Dozens of smaller studios pass through untouched by the `.get(key, s.strip())` fallback -- which is correct behavior, but it means *we have not proven they are clean*. In a real project you would profile the remaining values and keep extending the map. Cleaning is iterative, not one-and-done.

## 2.3 Parse dates

We have FIVE different formats in ONE column. `pd.to_datetime` with `format="mixed"` handles this well.

In [ ]:
df["release_date"] = pd.to_datetime(
    df["release_date"], format="mixed", dayfirst=False, errors="coerce"
)

print("Unparseable dates:", df["release_date"].isna().sum())
print(df["release_date"].head(10))

# Derive useful columns from the date -- the warehouse will want these
df["release_year"] = df["release_date"].dt.year
df["release_decade"] = (df["release_year"] // 10) * 10

df[["title", "release_date", "release_year", "release_decade"]].head()

> **`errors="coerce"` is important.** It turns anything unparseable into `NaT` (null) instead of crashing the whole pipeline. In production ETL you'd log those rows for review rather than silently dropping them.

## 2.4 Parse money columns

`$1,200,000`, `USD 50000`, `1200000` -- all the same number but different formatting on the currency


In [ ]:
def parse_money(v):
    if pd.isna(v) or str(v).strip() == "":
        return np.nan
    s = str(v)
    for junk in ["$", ",", "USD", " "]:
        s = s.replace(junk, "")
    try:
        return float(s)
    except ValueError:
        return np.nan

df["budget_usd"] = df["budget_usd"].apply(parse_money)
df["box_office_usd"] = df["box_office_usd"].apply(parse_money)

df[["title", "budget_usd", "box_office_usd"]].head(10)

## 2.5 Parse runtime

`110`, `97 min`, `1h 12m` -- notice there are three formats, but all mean the same thing if we convert to 'minutes'


In [ ]:
import re

def parse_runtime(v):
    if pd.isna(v) or str(v).strip() == "":
        return np.nan
    s = str(v).strip().lower()

    # "1h 12m" style
    hm = re.match(r"^(\d+)\s*h\s*(\d+)\s*m$", s)
    if hm:
        return int(hm.group(1)) * 60 + int(hm.group(2))

    # "97 min" / "97min" / "97"
    num = re.match(r"^(\d+)", s)
    if num:
        return int(num.group(1))

    return np.nan

df["runtime_min"] = df["runtime"].apply(parse_runtime)
df = df.drop(columns=["runtime"])

print(df["runtime_min"].describe())
df[["title", "runtime_min"]].head(10)

## 2.6 Numeric conversions


In [ ]:
df["user_score"] = pd.to_numeric(df["user_score"], errors="coerce")
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce")

df[["user_score", "vote_count"]].describe()

## 2.7 Handle missing values

There is no single right answer here -- the strategy depends on the column and on what the business needs.

In [ ]:
print("Missing before:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# Strategy 1: fill with a meaningful placeholder where "unknown" is a valid answer
df["director"] = df["director"].fillna("Unknown")
df["mpaa_rating"] = df["mpaa_rating"].fillna("Not Rated")

# Strategy 2: LEAVE numeric measures null.
# Do NOT fill budget with 0 -- that would silently corrupt every average.
# A missing budget means "we don't know", not "it cost nothing".

print("\nMissing after:")
print(df.isna().sum()[df.isna().sum() > 0])

> **This is a judgment call worth pausing on.** Filling a missing *category* with `"Unknown"` is usually fine -- it's honest, and it keeps the row usable. Filling a missing *number* with `0` is usually a bug: it turns "we don't know" into a confident false claim, and it drags every average down. When in doubt, leave numbers null and let the warehouse decide.

## 2.8 Deduplicate


In [ ]:
before = len(df)

# record_id is a surrogate from the source system, so ignore it when comparing
df = df.drop_duplicates(subset=[c for c in df.columns if c != "record_id"])

print(f"Removed {before - len(df)} duplicate rows ({before:,} -> {len(df):,})")

## 2.9 Split the multi-valued column

About a fifth of the rows have `subgenre` values like `Slasher|Occult`. That violates 1NF -- remember Nomralization!! In the warehouse this becomes its own bridge table.


In [ ]:
multi = df["subgenre"].str.contains("|", regex=False).sum()
print(f"Rows with multiple subgenres: {multi}")

# Build a separate long-format table: one row per (movie, subgenre)
subgenres = (
    df[["record_id", "subgenre"]]
    .assign(subgenre=df["subgenre"].str.split("|"))
    .explode("subgenre")
)
subgenres["subgenre"] = subgenres["subgenre"].str.strip()

print(f"\nMovie-subgenre pairs: {len(subgenres):,}")
subgenres.head(10)

> **`explode()` is the pandas equivalent of the 1NF fix** we did by hand in Lab 2. One row per value, joined back by the key.

## 2.10 Add derived columns and validate

Transformations often *create* value, not just clean it. When we say **derived** we mean this data has been created from something else.


In [ ]:
# A derived business metric
df["profit_usd"] = df["box_office_usd"] - df["budget_usd"]
df["roi"] = (df["box_office_usd"] / df["budget_usd"]).round(2)

df[["title", "budget_usd", "box_office_usd", "profit_usd", "roi"]].head(10)

In [ ]:
# Validation: does the cleaned data pass basic sanity checks?
checks = {
    "runtime between 40 and 300 min": df["runtime_min"].between(40, 300).all(),
    "user_score between 0 and 10":    df["user_score"].dropna().between(0, 10).all(),
    "release_year is plausible":      df["release_year"].dropna().between(1890, 2030).all(),
    "no negative budgets":            (df["budget_usd"].dropna() >= 0).all(),
    "no duplicate record_ids":        df["record_id"].is_unique,
}

for name, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {name}")

> **Always validate at the end of Transform.** A pipeline that silently loads garbage is worse than one that crashes -- at least a crash tells you something is wrong.

# Part 3: Loading

Now we write the clean data into SQL Server.

## 3.1 Connect

In [ ]:
import pyodbc
print(pyodbc.drivers())

If you do not see `'ODBC Driver 18 for SQL Server'` in the list that the above statement generates, run `winget install Microsoft.msodbcsql.18` on Powershell (Windows) or run the following two commands on MacOS (you'll need Homebrew).

- `brew tap microsoft/mssql-release https://github.com/Microsoft/homebrew-mssql-release`
- `brew install msodbcsql18`


In [ ]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Adjust the port/password to match your container
params = quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost,1434;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=Passw0rd1!;"
    "TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as conn:
    print("Connected.")

## 3.2 Staging vs. final tables

Professional ETL rarely writes straight into the production table. Instead:

1. Load into a **staging table** -- a scratch area shaped like the incoming data.
2. Validate and reconcile there.
3. Move it into the **final table** in a controlled step.

Why bother? Because if the load fails halfway through, your real table is untouched. It's the same instinct as a transaction.


In [ ]:
from sqlalchemy import text

# CREATE DATABASE cannot run inside a transaction — use autocommit
with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("IF DB_ID('HorrorDW') IS NULL CREATE DATABASE HorrorDW;"))

# reconnect pointing at the new database
params_dw = params.replace(quote_plus("DATABASE=master"), quote_plus("DATABASE=HorrorDW"))
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_dw}")

# schemas are fine inside a transaction
with engine.begin() as conn:
    conn.execute(text("IF SCHEMA_ID('stg') IS NULL EXEC('CREATE SCHEMA stg');"))
    conn.execute(text("IF SCHEMA_ID('dw')  IS NULL EXEC('CREATE SCHEMA dw');"))

print("HorrorDW ready with stg and dw schemas.")

## 3.3 Load into staging


In [ ]:
load_cols = [
    "record_id", "title", "release_date", "release_year", "release_decade",
    "director", "studio", "country", "runtime_min", "mpaa_rating",
    "budget_usd", "box_office_usd", "profit_usd", "roi",
    "user_score", "vote_count",
]

df[load_cols].to_sql(
    "movies", con=engine, schema="stg",
    if_exists="replace",   # truncate-and-reload
    index=False, chunksize=500
)

print(f"Loaded {len(df):,} rows into stg.movies")

In [ ]:
# Load the subgenre bridge too
subgenres.to_sql(
    "movie_subgenre", con=engine, schema="stg",
    if_exists="replace", index=False, chunksize=1000
)

print(f"Loaded {len(subgenres):,} rows into stg.movie_subgenre")

## 3.4 Verify the load

Never trust a load you haven't checked.


In [ ]:
check = pd.read_sql("""
    SELECT COUNT(*)                AS row_count,
           COUNT(DISTINCT studio)  AS distinct_studios,
           MIN(release_year)       AS earliest,
           MAX(release_year)       AS latest,
           AVG(CAST(user_score AS FLOAT)) AS avg_score
    FROM stg.movies
""", con=engine)

check

In [ ]:
# Does the row count match what we sent?
sql_count = pd.read_sql("SELECT COUNT(*) AS n FROM stg.movies", con=engine)["n"][0]
print(f"pandas rows: {len(df):,}")
print(f"SQL rows:    {sql_count:,}")
print("MATCH" if sql_count == len(df) else "MISMATCH -- investigate!")

## 3.5 Full-reload vs. incremental

We just did a **full reload**: drop everything, write it all again. That's simple and fine for a dataset this size.

For a table with 50 moillion rows and a daily feed, you'd do an **incremental load** instead -- only lookig at the rows that are new or changed since last time:

- Track a **watermark** (e.g. the max `last_modified` you've already loaded).
- Extract only rows newer than that watermark.
- **Upsert** them: update rows that already exist, insert the ones that don't (T-SQL's `MERGE`, or a delete-then-insert).

| | Full reload | Incremental |
|---|---|---|
| Simplicity | Very simple | More moving parts |
| Runtime | Grows with total size | Grows with *change* size |
| Handles deletes | Automatically | Needs explicit handling |
| Good for | Small/medium tables | Large tables, frequent runs |

---

# Part 4 -- Where this goes next

You now have clean, conformed data in `stg.movies` and `stg.movie_subgenre`. That is the *input* to a data warehouse, not the warehouse itself.

Look at what's sitting in that staging table and notice something: the columns naturally fall into two kinds.

- Things you'd **measure or add up**: budget, box office, profit, ROI, vote count, score.
- Things you'd **slice or filter by**: studio, country, director, release year, rating, subgenre.

This distinction - **facts** versus **dimensions** - is the foundation of __dimensional modeling__, and it's exactly where we pick up on Thursday. We'll take this same data and reshape `stg.movies` into a proper star schema.


In [ ]:
# A preview of the question Thursday answers: which columns are which?
measures  = ["budget_usd", "box_office_usd", "profit_usd", "roi", "user_score", "vote_count"]
attributes = ["studio", "country", "director", "release_year", "release_decade", "mpaa_rating"]

print("Candidate FACTS (things we measure):")
for m in measures: print("   ", m)

print("\nCandidate DIMENSIONS (things we slice by):")
for a in attributes: print("   ", a)

---

## Summary

- **ETL** = Extract, Transform, Load. **ELT** flips the last two and cleans inside the warehouse.
- **Extract raw** - read everything as text so nothing is silently mis-parsed.
- **Profile before you clean.** Count nulls, look at distinct values, eyeball the formats.
- **Standardization** (mapping `universal` -> `Universal`) is the highest-value cleaning step; without it every `GROUP BY` lies.
- **Don't fill missing numbers with zero.** Unknown is not the same as none.
- **Deduplicate**, split multi-valued columns (1NF again), derive new metrics, then **validate**.
- **Stage first, then promote.** Don't write straight into production.
- **Verify the load** by comparing counts and spot-checking aggregates.
- Full reload is simple; **incremental** loads scale.

### Knowldge Check -- Are you Good?

1. Why do we read the CSV with `dtype=str` instead of letting pandas infer types?
2. What does `errors="coerce"` do, and what's the risk of using it carelessly?
3. Why is filling a missing `budget_usd` with `0` a bad idea when filling a missing `director` with `"Unknown"` is fine?
4. What problem does a staging table solve?
5. Give one column from this dataset that is clearly a *fact* and one that is clearly a *dimension*.
